In [ ]:
# imports

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset
from pinecone import Pinecone
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer



# setup
load_dotenv()
pine_client = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
pine_index = pine_client.Index("humanitarian-risk")
llm = ChatAnthropic(model="claude-haiku-4-5-20251001")
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
)

chat_history = []

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a humanitarian data analyst assistant.
Answer questions based only on the provided context.
Be precise with numbers and facts.
Context: {context}"""),
    ("human", "{question}")
])




class Question(BaseModel):
    question: str
    

# building retriever function -that takes a question, searches ChromaDB
# return top 3 relevant documents from each collection and combining them
# loading vectors back from Pinecone to check
def retrieve_context(question):
    query_embedding = embed_model.encode(question).tolist()

    food_results = pine_index.query(
        vector=query_embedding,
        namespace="food_prices",
        top_k=9,
        include_metadata=True
    )

    poverty_results = pine_index.query(
        vector=query_embedding,
        namespace="poverty_mpi",
        top_k=9,
        include_metadata=True
    )
    
    food_texts= [match['metadata']['text'] for match in food_results['matches']]
    poverty_texts = [match['metadata']['text'] for match in poverty_results['matches']]
    all_texts = food_texts + poverty_texts
    context = "\n\n".join(all_texts)
    return context



My_sample_questions= [
"what is poverty situation in uttar pradesh?",
"what is historical food price trends in uttar pradesh?",
"what is relationship between poverty and price of rice and wheat in uttar pradesh?",
"how is the risk of hunger and starvation in uttar pradesh?"
]


answers= []
contexts=[]

chain = prompt | llm


for question in My_sample_questions:
    context = retrieve_context(question)
    contexts.append([context])
    answer = chain.invoke({"context": context, "question": question})
    answers.append(answer.content)

# RAGAS dataset
data= {
    "question": My_sample_questions,
    "answer": answers,
    "contexts":contexts,
    "ground_truth": [
    # Q1 - poverty situation
    "Uttar Pradesh has significant multidimensional poverty. 68.79% of the population lives in poverty with an MPI score of 0.3611. Other measures show 40.69% poverty at MPI 0.1821 and 22.94% at MPI 0.0983.",
    
    # Q2 - historical food price trends
    "Rice retail prices in Uttar Pradesh doubled from 8.0 INR per KG in February 1999 to 16.0 INR per KG by January 2010, remaining stable at 16.0 INR per KG throughout 2010 and 2011.",
    
    # Q3 - relationship between poverty and food prices
    "68.79% of Uttar Pradesh population lives in poverty. Rice prices doubled from 8.0 INR/KG in 1999 to 16.0 INR/KG in 2010. Wheat price data is not available. Direct causal relationship cannot be established due to non-overlapping time periods.",
    
    # Q4 - hunger and starvation risk
    "Uttar Pradesh faces significant hunger risk with 68.79% population in multidimensional poverty. Rice prices doubled from 8.0 INR/KG to 16.0 INR/KG between 1999 and 2010. No direct hunger or malnutrition indicators available in the dataset."
]
}
dataset= Dataset.from_dict(data)
# evaluate
results= evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm= ragas_llm,
    embeddings=ragas_embeddings
)
print(results)
df= results.to_pandas()
df

g:\Projects\venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

# RAGAS Evaluation Report — Project 1 RAG System
## Date: June 2026

---

## Evaluation Run 1 — Vague Ground Truths (Baseline)

| Metric | Score | Rating |
|---|---|---|
| Faithfulness | 0.7231 | Acceptable |
| Answer Relevancy | 0.3921 | Needs Improvement |
| Context Precision | 0.7500 | Good |
| Context Recall | 0.2500 | Poor |

---

## Evaluation Run 2 — Specific Ground Truths (Improved)

| Metric | Score | Rating |
|---|---|---|
| Faithfulness | 0.6314 | Needs Improvement |
| Answer Relevancy | 0.4098 | Needs Improvement |
| Context Precision | 0.2500 | Poor |
| Context Recall | 0.3750 | Poor |

---

## Per-Question Breakdown — Run 2

| Question | Faithfulness | Answer Relevancy | Context Precision | Context Recall |
|---|---|---|---|---|
| Poverty situation in UP | 0.571 | 0.784 | 1.0 | 0.667 |
| Historical food price trends in UP | 0.938 | 0.855 | 0.0 | 0.000 |
| Relationship between poverty and rice/wheat price | 0.600 | 0.000 | 0.0 | 0.500 |
| Risk of hunger and starvation in UP | 0.417 | 0.000 | 0.0 | 0.333 |

---

## Key Findings

**Finding 1 — Retrieval biased toward food price data (Critical)**
- All 4 questions retrieved rice price chunks regardless of query type
- Poverty MPI chunks are not being surfaced for poverty-specific questions
- Context precision dropped from 0.75 to 0.25 with specific ground truths —
  exposing that retrieved chunks don't match what was actually needed
- Root cause: food_prices namespace dominates retrieval across both namespaces

**Finding 2 — Complex cross-dataset questions completely fail (Critical)**
- Q3 (poverty + food price relationship) and Q4 (hunger risk) scored 0.0 
  on answer_relevancy — LLM could not answer due to insufficient context
- These questions require simultaneous retrieval from both namespaces 
  with sufficient depth — current top_k=9 per namespace is insufficient

**Finding 3 — Food price questions work well in isolation (Strength)**
- Q2 (historical food price trends) scored faithfulness=0.938, 
  answer_relevancy=0.855 — highest scores across both runs
- Single-dataset, single-namespace queries work reliably

**Finding 4 — Poverty questions partially work (Moderate)**
- Q1 (poverty situation) context_recall=0.667 — retrieving some 
  but not all needed poverty chunks
- Faithfulness=0.571 — LLM hallucinating beyond retrieved context 
  for poverty questions

**Finding 5 — Ground truth quality matters (Methodology)**
- Vague ground truths (Run 1) inflated context_precision artificially
- Specific ground truths (Run 2) gave more accurate diagnostic picture
- Always use data-grounded specific ground truths for meaningful RAGAS scores

---

## Root Cause Analysis

Retrieval Issue:

food_prices namespace → 9 chunks retrieved

poverty_mpi namespace → 9 chunks retrieved

→ But food price chunks are semantically closer to ALL queries

→ Poverty chunks losing relevance competition in combined retrieval

Result:

- Food price questions: good retrieval → good answers
- Poverty questions: wrong chunks retrieved → poor answers
- Cross-dataset questions: neither namespace retrieved adequately → fails

---

## Identified Improvements

1. **Namespace-aware retrieval** (Priority: High)
   - Detect query type before retrieval
   - Route poverty questions → query poverty_mpi namespace with higher top_k
   - Route food price questions → query food_prices namespace with higher top_k
   - Route cross-dataset questions → increase top_k on both namespaces

2. **Increase top_k for complex queries** (Priority: High)
   - Current: top_k=9 per namespace
   - Proposed: top_k=15-20 for cross-dataset questions
   - Expected impact: improved context recall for Q3 and Q4

3. **Tighten generation prompt** (Priority: Medium)
   - Stronger "only use provided context" instruction
   - Explicit instruction: "state what data is missing rather than inferring"
   - Expected impact: improved faithfulness on poverty questions

4. **Chunking strategy review** (Priority: Low)
   - Fixed-size chunking may split related data across chunks
   - Consider larger chunk sizes for poverty MPI records
   - Consider semantic chunking for better boundary detection